DATA CLEANING

Being Our PySpark Session.

In [17]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder.appName("Data Cleaning")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config('spark.driver.memory', '8g')
    .config('spark.executor.memory', '4g')
    .getOrCreate()
)
# Silence any WARN-level logs from this point forth.
spark.sparkContext.setLogLevel("ERROR")

In [18]:
sdf = spark.read.csv(data_file, header=True)

sdf.printSchema()
sdf.show(1, vertical=True)

sdf.select("land_area").filter(F.col("land_area").isNotNull()).show(20, truncate=False)

root
 |-- listing_id: string (nullable = true)
 |-- suburb: string (nullable = true)
 |-- postcode: string (nullable = true)
 |-- weekly_rent: string (nullable = true)
 |-- bond: string (nullable = true)
 |-- available_date: string (nullable = true)
 |-- date_listed: string (nullable = true)
 |-- days_listed: string (nullable = true)
 |-- bedrooms: string (nullable = true)
 |-- bathrooms: string (nullable = true)
 |-- carspaces: string (nullable = true)
 |-- property_type: string (nullable = true)
 |-- address: string (nullable = true)
 |-- lat: string (nullable = true)
 |-- lon: string (nullable = true)
 |-- scraped_date: string (nullable = true)
 |-- domain_page_id: string (nullable = true)
 |-- property_id: string (nullable = true)
 |-- photo_count: string (nullable = true)
 |-- video_count: string (nullable = true)
 |-- floorplans_count: string (nullable = true)
 |-- virtual_tour: string (nullable = true)
 |-- primary_type: string (nullable = true)
 |-- secondary_type: string (null

Define Our Schema to tackle any schema level errors.

In [5]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    DateType,
    TimestampType,
    BooleanType,
)

rental_listings_schema = StructType([
    StructField("listing_id", StringType(), True),
    StructField("suburb", StringType(), True),
    StructField("postcode", StringType(), True),
    StructField("weekly_rent", DoubleType(), True),
    StructField("bond", DoubleType(), True),
    StructField("available_date", StringType(), True),
    StructField("date_listed", DateType(), True),
    StructField("days_listed", DoubleType(), True),
    StructField("bedrooms", DoubleType(), True),
    StructField("bathrooms", DoubleType(), True),
    StructField("carspaces", DoubleType(), True),
    StructField("property_type", StringType(), True),
    StructField("address", StringType(), True),
    StructField("lat", DoubleType(), True),
    StructField("lon", DoubleType(), True),
    StructField("scraped_date", TimestampType(), True),
    StructField("domain_page_id", StringType(), True),
    StructField("property_id", StringType(), True),
    StructField("photo_count", DoubleType(), True),
    StructField("video_count", DoubleType(), True),
    StructField("floorplans_count", DoubleType(), True),
    StructField("virtual_tour", BooleanType(), True),
    StructField("primary_type", StringType(), True),
    StructField("secondary_type", StringType(), True),
    StructField("agency", StringType(), True),
    StructField("agency_id", DoubleType(), True),
    StructField("agent_names", StringType(), True),
    StructField("structured_features", StringType(), True),
    StructField("url", StringType(), True),
    StructField("land_area", StringType(), True),
])

# Example usage:
# df = spark.read.schema(rental_listings_schema).csv("path/to/file.csv", header=True)

In [7]:
data_file = "rea_data/domain/Data/vic_rentals_all.csv"
sdf = spark.read.csv(data_file, header=True)

sdf.printSchema()

root
 |-- listing_id: string (nullable = true)
 |-- suburb: string (nullable = true)
 |-- postcode: string (nullable = true)
 |-- weekly_rent: string (nullable = true)
 |-- bond: string (nullable = true)
 |-- available_date: string (nullable = true)
 |-- date_listed: string (nullable = true)
 |-- days_listed: string (nullable = true)
 |-- bedrooms: string (nullable = true)
 |-- bathrooms: string (nullable = true)
 |-- carspaces: string (nullable = true)
 |-- property_type: string (nullable = true)
 |-- address: string (nullable = true)
 |-- lat: string (nullable = true)
 |-- lon: string (nullable = true)
 |-- scraped_date: string (nullable = true)
 |-- domain_page_id: string (nullable = true)
 |-- property_id: string (nullable = true)
 |-- photo_count: string (nullable = true)
 |-- video_count: string (nullable = true)
 |-- floorplans_count: string (nullable = true)
 |-- virtual_tour: string (nullable = true)
 |-- primary_type: string (nullable = true)
 |-- secondary_type: string (null

In [13]:
cols_drop = ["listing_id", "property_id", "domain_page_id", "url", 
             "scraped_date", "photo_count", "video_count", "floorplans_count",
             "virtual_tour", "agency", "agency_id", "agent_names", "bond",
             "land_area"]

sdf = sdf.drop(*cols_drop)

In [14]:
for column in sdf.columns:
    missing_val = sdf.filter(F.col(column).isNull()).count()
    print(f"{column}: {missing_val}")

suburb: 14
postcode: 20
weekly_rent: 299
available_date: 157
date_listed: 25
days_listed: 25
bedrooms: 146
bathrooms: 72
carspaces: 1824
property_type: 21
address: 125
lat: 25
lon: 25
primary_type: 25
secondary_type: 25
structured_features: 1347


In [16]:
missing_location = sdf.filter(F.col("suburb").isNull() | F.col("postcode").isNull())

missing_location.select("suburb", "postcode", "address").show(50, truncate=False)

+------------------------------------------------------------------------------+--------+-------+
|suburb                                                                        |postcode|address|
+------------------------------------------------------------------------------+--------+-------+
|NULL                                                                          |NULL    |NULL   |
|https://www.domain.com.au/312-673-la-trobe-street-docklands-vic-3008-15802858 |NULL    |NULL   |
|NULL                                                                          |NULL    |NULL   |
|https://www.domain.com.au/312-673-la-trobe-street-docklands-vic-3008-17747866 |NULL    |NULL   |
|NULL                                                                          |NULL    |NULL   |
|NULL                                                                          |NULL    |NULL   |
|NULL                                                                          |NULL    |NULL   |
|https://www.domain.

In [18]:
sdf.select([F.min(c).alias(f"{c}_min") for c in sdf.columns] + 
           [F.max(c).alias(f"{c}_max") for c in sdf.columns]).show(truncate=False)

+-------------------------------------------------------------------------------+-------------------------+---------------+---------------------+---------------+---------------+------------+-------------+-------------+--------------------+-----------------------+---------+---------+----------------+--------------------+-----------------------+-----------------------------------------------------------------------+------------+---------------------------------------------------------------------------+-----------------------+---------------+---------------+------------+-------------+-------------+-----------------+----------------------+---------+--------+----------------+------------------+--------------------------------+
|suburb_min                                                                     |postcode_min             |weekly_rent_min|available_date_min   |date_listed_min|days_listed_min|bedrooms_min|bathrooms_min|carspaces_min|property_type_min   |address_min            |lat_

In [6]:
sdf.count()

12738